In [3]:
# Install all required libraries
%pip install pandas scikit-learn catboost xgboost seaborn matplotlib joblib openai faiss-cpu torch python-dotenv tqdm pyarrow fastparquet

# Note: You must upload 'car_prices_extended_eda.csv' and 'News_dataset.csv'
# to your Colab session, or mount Google Drive to access them.

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import os
import faiss
from tqdm import tqdm
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor

In [10]:
# --- Configuration ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
EMBEDDING_MODEL = "text-embedding-3-small"
DEFAULT_DIMENSION = 1536
BATCH_SIZE = 1000
D = DEFAULT_DIMENSION # Dimension for FAISS indexing

# --- Define File Paths ---
NEWS_INDEX_FILE = 'faiss_news_index.cbm'
CAR_EMBEDDINGS_FILE = 'car_embeddings_numpy.npy'
NEWS_METADATA_FILE_PKL = 'news_metadata.pkl'
CAR_DATA_FINAL_FILE_PKL = 'car_data_final.pkl'

In [23]:
# --- Load Saved Data ---
print("Loading saved data...")

# Load Car Data (contains all non-embedding features)
# This creates the 'df' DataFrame
df = pd.read_pickle(CAR_DATA_FINAL_FILE_PKL)
df['sale_date'] = pd.to_datetime(df[['sale_year', 'sale_month', 'sale_day']].rename(columns={'sale_year': 'year', 'sale_month': 'month', 'sale_day': 'day'}))
print(f"Car data loaded. Shape: {df.shape}")

# Load Car Embeddings (the 'embeddings' array)
embeddings = np.load(CAR_EMBEDDINGS_FILE)
embedding_cols = [f'e_{i}' for i in range(D)]

# **CRITICAL FIX**: Recreate 'df_embeddings' as a standalone DataFrame 
# This is what the RAG cell *should* have been referencing instead of 'car_embeddings_df'
df_embeddings = pd.DataFrame(embeddings, columns=embedding_cols, index=df.index)

# Load News Metadata
# This creates the 'NEWS_DF' DataFrame
NEWS_DF = pd.read_pickle(NEWS_METADATA_FILE_PKL)
print(f"News metadata loaded. Shape: {NEWS_DF.shape}")

# Load News Embeddings (the 'news_embeddings' array)
# This is required by the `retrieve_news_features_optimized` function
news_embeddings = np.load('news_embeddings_numpy.npy')
print(f"News embeddings loaded. Shape: {news_embeddings.shape}")

# Load FAISS Index
# This creates the 'news_index' object
news_index = faiss.read_index(NEWS_INDEX_FILE)
print(f"FAISS index loaded. Total vectors: {news_index.ntotal}")

print(f"All car features (including embeddings) combined. Final df shape: {df.shape}")

Loading saved data...
Car data loaded. Shape: (472325, 1569)
News metadata loaded. Shape: (209527, 4)
News embeddings loaded. Shape: (209527, 1536)
FAISS index loaded. Total vectors: 209527
All car features (including embeddings) combined. Final df shape: (472325, 1569)


In [12]:

def get_embeddings_batch(client, texts, model=EMBEDDING_MODEL):
    """Fetches a batch of embeddings from the OpenAI API."""
    try:
        response = client.embeddings.create(input=texts, model=model)
        return [d.embedding for d in response.data]
    except Exception as e:
        # Handle API errors gracefully by returning zero vectors
        print(f"API Error during batch. Error: {e}")
        return [[0.0] * DEFAULT_DIMENSION] * len(texts)

def build_ivfpq_index(embeddings_array, name="Index"):
    """
    Builds a memory-efficient FAISS IndexIVFPQ.
    Requires the embeddings array to be float32.
    """
    N_VEC = embeddings_array.shape[0]
    NLIST = 100
    M = 64
    N_TRAIN = min(20000, N_VEC) # Use a subset for training

    # 1. Create a coarse quantizer
    quantizer = faiss.IndexFlatL2(D)

    # 2. Create the IndexIVFPQ index
    # M=64 means the vector storage is significantly compressed (1536*4 bytes to 64 bytes)
    index = faiss.IndexIVFPQ(quantizer, D, NLIST, M, 8)

    # 3. Train the index
    print(f"Training {name} IndexIVFPQ using {N_TRAIN} vectors...")
    index.train(embeddings_array[:N_TRAIN])

    # 4. Add the full vector set
    print(f"Adding {N_VEC} vectors to the trained {name} index...")
    index.add(embeddings_array)
    print(f"FAISS {name} IndexIVFPQ built with {index.ntotal} compressed vectors.")
    return index

In [3]:
def setup_client():
    """Initializes OpenAI Client and loads environment variables."""
    # Load environment variables from .env file (if uploaded to Colab session)
    load_dotenv()
    print("Environment variables from .env file loaded.")

    try:
        # Initialize OpenAI Client (will use OPENAI_API_KEY)
        client = OpenAI()
        print("OpenAI Client initialized successfully.")
        return client
    except Exception as e:
        print("ERROR: Failed to initialize OpenAI client. Ensure 'OPENAI_API_KEY' environment variable is set.")
        raise e

# Setup
client = setup_client()

# Load Car Data
print("\nLoading Car Data...")
df = pd.read_csv("car_prices_extended_eda.csv")

# Create composite text embedding for car features
df['text_embedding'] = df.apply(
    lambda row: f"Car: {row['make']} {row['model']} {row['trim']} {row['body']} {row['car_age']} {row['condition_category']} {row['transmission']} {row['seller_category']} {row['is_luxury']}, Year: {row['year']}, Mileage: {row['odometer']} {row['mileage_per_year']} {row['model']}, Price: {row['mmr']} {row['sellingprice']} {row['price_diff']} {row['price_gap_pct']}, Sale date: {row['sale_year']} {row['sale_month']} {row['sale_day']} {row['is_weekend']}",
    axis=1
)
# Create the sale_date column, essential for time-sensitive RAG
df['sale_date'] = pd.to_datetime(df[['sale_year', 'sale_month', 'sale_day']].rename(columns={'sale_year': 'year', 'sale_month': 'month', 'sale_day': 'day'}))
print(f"Car data loaded with {len(df)} records.")

Environment variables from .env file loaded.
OpenAI Client initialized successfully.

Loading Car Data...
Car data loaded with 472325 records.


In [4]:
print("\n--- Car Embedding Generation (Batch Processed) ---")
all_car_texts = df['text_embedding'].tolist()
car_batched_embeddings = []

for i in tqdm(range(0, len(all_car_texts), BATCH_SIZE), desc="Generating Car Embeddings"):
    batch = all_car_texts[i:i + BATCH_SIZE]
    car_batched_embeddings.extend(get_embeddings_batch(client, batch))

embeddings = np.array(car_batched_embeddings, dtype='float32')

# Convert Car Embeddings into CatBoost features (e_0, e_1, ...)
embedding_cols = [f'e_{i}' for i in range(D)]
df_embeddings = pd.DataFrame(embeddings, columns=embedding_cols, index=df.index)
df = pd.concat([df, df_embeddings], axis=1)
print(f"Car embeddings added as {D} features to the DataFrame.")


--- Car Embedding Generation (Batch Processed) ---


Generating Car Embeddings: 100%|█████████████████████████████████████████████████████| 473/473 [43:34<00:00,  5.53s/it]


Car embeddings added as 1536 features to the DataFrame.


In [5]:
df_news = pd.read_csv("News_dataset.csv")

In [6]:
print("\n--- News Data Processing ---")
# Preprocessing
df_news = df_news.drop(columns=['Unnamed: 0'], errors='ignore')
df_news[['headline', 'short_description']] = df_news[['headline', 'short_description']].fillna('')
df_news['news_date'] = pd.to_datetime(df_news[['year', 'month', 'day']])
df_news['text_for_embedding'] = df_news['headline'] + ' | ' + df_news['short_description']
print(f"News dataset loaded with {len(df_news)} articles.")


--- News Data Processing ---
News dataset loaded with 209527 articles.


In [5]:
print("\n--- News Embedding, and Indexing ---")

# News Embedding Generation (Batch Processed)
all_news_texts = df_news['text_for_embedding'].tolist()
# This list will now store smaller NumPy array chunks, not Python lists
news_embedding_chunks = [] 

print("Starting batch API calls for News embeddings...")

for i in tqdm(range(0, len(all_news_texts), BATCH_SIZE), desc="Generating News Embeddings"):
    batch = all_news_texts[i:i + BATCH_SIZE]
    list_of_embeddings = get_embeddings_batch(client, batch)
    
    # Convert the small batch list directly to a NumPy array chunk and store it
    chunk = np.array(list_of_embeddings, dtype='float32') 
    news_embedding_chunks.append(chunk)

# Use np.vstack to concatenate all the small NumPy chunks into the final array
# This is much more memory efficient than np.array() on a massive Python list
news_embeddings = np.vstack(news_embedding_chunks)

# Build News FAISS Index (IndexIVFPQ)
NEWS_DF = df_news
news_index = build_ivfpq_index(news_embeddings, name="News")


--- News Data Processing, Embedding, and Indexing ---
News dataset loaded with 209527 articles.
Starting batch API calls for News embeddings...


Generating News Embeddings: 100%|████████████████████████████████████████████████████| 210/210 [20:08<00:00,  5.75s/it]


Training News IndexIVFPQ using 20000 vectors...
Adding 209527 vectors to the trained News index...
FAISS News IndexIVFPQ built with 209527 compressed vectors.


In [6]:
# --- 1. Save News Index (The Vector Store) ---
# Assuming 'news_index' and 'NEWS_DF' (which points to df_news) are available globally
try:
    faiss.write_index(news_index, NEWS_INDEX_FILE)
    print(f"FAISS News Index saved: {NEWS_INDEX_FILE}")
except NameError:
    print("Error: 'news_index' not found. Ensure Section 5 was run successfully.")

NEWS_EMBEDDINGS_FILE = 'news_embeddings_numpy.npy'
np.save(NEWS_EMBEDDINGS_FILE, news_embeddings)
print(f"Raw News Embeddings (NumPy) saved: {NEWS_EMBEDDINGS_FILE}")

FAISS News Index saved: faiss_news_index.cbm
Raw News Embeddings (NumPy) saved: news_embeddings_numpy.npy


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [14]:
try:
    # Switched to Pickle (.pkl) to avoid PyArrow extension errors
    NEWS_DF.drop(columns=['text_for_embedding', 'year', 'month', 'day'], errors='ignore').to_pickle(NEWS_METADATA_FILE_PKL)
    print(f"News Metadata saved: {NEWS_METADATA_FILE_PKL}")
except NameError:
    print("Error: 'NEWS_DF' not found.")


News Metadata saved: news_metadata.pkl


In [15]:
# --- 4. Save Final Car Data DataFrame (Using Pickle) ---
try:
    # Switched to Pickle (.pkl) to avoid PyArrow extension errors
    df.drop(columns=['text_embedding', 'sale_date'], errors='ignore').to_pickle(CAR_DATA_FINAL_FILE_PKL)
    print(f"Final Car Data (with all features) saved: {CAR_DATA_FINAL_FILE_PKL}")
except NameError:
    print("Error: 'df' not found.")

Final Car Data (with all features) saved: car_data_final.pkl


In [16]:
# --- 3. Save Car Embeddings (NumPy Array) ---
# This is the vector array used for the RAG query and CatBoost features.
try:
    np.save(CAR_EMBEDDINGS_FILE, embeddings)
    print(f"Car Embeddings (NumPy) saved: {CAR_EMBEDDINGS_FILE}")
except NameError:
    print("Error: 'embeddings' not found. Ensure Section 4 was run successfully.")

Car Embeddings (NumPy) saved: car_embeddings_numpy.npy


In [14]:
### 5. Time-Sensitive RAG Feature Generation (CORRECTED) ###

K = 5 # Number of nearest neighbors to retrieve from the News Index
TIME_WINDOW_DAYS = 90
# NEWS_DF is expected to be loaded from the previous step

# We need to redefine the function to accept the sale date directly
def retrieve_news_features_optimized(sale_date_obj, car_embedding):
    """Retrieves and aggregates features from the news index based on time and semantic relevance."""
    try:
        # 1. Search the news index for K nearest neighbors
        # news_index is expected to be loaded from the previous step
        D_dist, I_ids = news_index.search(car_embedding.reshape(1, -1), K)
        news_indices = I_ids.flatten()

        # 2. Define the relevant time window
        sale_date = sale_date_obj
        time_cutoff = sale_date - pd.Timedelta(days=TIME_WINDOW_DAYS)

        # 3. Find the index of the nearest, time-valid article
        for original_idx in news_indices:
            # NEWS_DF is expected to be available
            news_date = NEWS_DF.iloc[original_idx]['news_date']
            
            # Check if news is relevant (before sale_date and within time_cutoff)
            if news_date < sale_date and news_date >= time_cutoff:
                # news_embeddings is expected to be loaded
                return news_embeddings[original_idx]
             
    except Exception as e:
        # print(f"RAG error: {e}") 
        pass

    # If no relevant news is found (or on error), return a zero vector
    # D is expected to be available (1536)
    return np.zeros(D, dtype='float32')


print("Starting RAG feature generation (memory-optimized)...")

rag_feature_list = []
# FIX 1: Use df_embeddings which was correctly created/loaded, not 'car_embeddings_df'
# df_embeddings is expected to be loaded/defined from the previous step
car_embeddings_rows = df_embeddings.values
sale_dates = df['sale_date'].values # df is expected to be loaded/defined

# Iterate using zip over the required arrays
for sale_date_obj, car_embedding in tqdm(
    zip(sale_dates, car_embeddings_rows), 
    total=len(df), 
    desc="RAG Feature Generation"
):
    rag_feature = retrieve_news_features_optimized(sale_date_obj, car_embedding)
    rag_feature_list.append(rag_feature)


# FIX 2: Move array construction and DataFrame operations OUTSIDE the loop

# Use np.vstack to combine the list of NumPy arrays efficiently
rag_features = np.vstack(rag_feature_list) 
print(f"\nSuccessfully constructed final RAG features array with shape: {rag_features.shape}")

# Add RAG features to the main DataFrame
rag_cols = [f'rag_e_{i}' for i in range(rag_features.shape[1])]
df_rag = pd.DataFrame(rag_features, columns=rag_cols, index=df.index)
df = pd.concat([df, df_rag], axis=1)

print(f"RAG features added: {rag_features.shape[1]} new columns to DataFrame 'df'.")

# You can now proceed to the CatBoost training step.

Starting RAG feature generation (memory-optimized)...


RAG Feature Generation: 100%|█████████████████████████████████████████████████| 472325/472325 [12:31<00:00, 628.14it/s]



Successfully constructed final RAG features array with shape: (472325, 1536)
RAG features added: 1536 new columns to DataFrame 'df'.


In [17]:
# --- 1. Clean up duplicate columns ---
# This ensures that if the concatenation cells were run multiple times, 
# we remove any columns that are duplicates, keeping the first occurrence.
df = df.loc[:, ~df.columns.duplicated(keep='first')]

# --- 2. Define Features and Target (as done in the original notebook) ---
# Assuming 'Sale_Price_Log' is your target
TARGET = 'log_sellingprice'
FEATURES_TO_EXCLUDE = [
    TARGET, 
    'sale_date', 
    'Title', 
    # Add any other non-feature columns here
]

# Get the final feature list
features = [col for col in df.columns if col not in FEATURES_TO_EXCLUDE]

# --- 3. Split the Data ---
X = df[features]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# --- 4. Identify Categorical Features (Crucial for CatBoost) ---
# Find the indices of non-numeric columns in the FINAL feature set (X_train)
categorical_features_indices = np.where(X_train.dtypes != np.float64)[0].tolist()

print(f"X_train shape after cleanup: {X_train.shape}")
print(f"Number of categorical features: {len(categorical_features_indices)}")
print("Data is ready for CatBoost training.")

X_train shape after cleanup: (377860, 3104)
Number of categorical features: 3097
Data is ready for CatBoost training.


In [22]:
X_train

,year,make,model,trim,body,transmission,vin,state,condition,odometer,...,rag_e_1526,rag_e_1527,rag_e_1528,rag_e_1529,rag_e_1530,rag_e_1531,rag_e_1532,rag_e_1533,rag_e_1534,rag_e_1535
34327,2013,Lexus,RX 350,Base,suv,automatic,2t2bk1ba5dc175155,tx,49.0,46887.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7869,2011,Infiniti,G Convertible,G37,convertible,automatic,jn1cv6fe4bm950736,fl,25.0,33309.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
469646,2014,Nissan,Rogue,SV,suv,automatic,5n1at2mt3ec760520,tx,42.0,18701.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
153557,2012,Honda,Accord,EX-L,sedan,automatic,1hgcp2f81ca094178,pa,36.0,33574.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
468890,2009,Hyundai,Elantra,SE,sedan,automatic,kmhdu46d89u798952,fl,35.0,75696.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259178,2012,Nissan,Altima,2.5 S,sedan,automatic,1n4al2ap2cn521270,mo,47.0,34183.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
365838,2006,Lexus,IS 250,Base,sedan,automatic,jthbk262462001803,ca,35.0,133159.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
131932,2013,Infiniti,G Sedan,G37x,sedan,automatic,jn1cv6ar5dm760473,la,41.0,39710.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
146867,2006,Ford,Fusion,SEL,sedan,automatic,3fahp08116r140875,nc,25.0,163172.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
print("\n--- CatBoost Training and Evaluation ---")

# 1. Target Variable Transformation
df['log_sellingprice'] = np.log1p(df['sellingprice'])

# 2. Define Features and Split Data
EXCLUDE_COLS = [
    'text_embedding', 'sellingprice', 'log_sellingprice', 'mmr',
    'price_diff', 'price_gap_pct', 'sale_date'
]

X = df.drop(columns=EXCLUDE_COLS, errors='ignore')
y = df['log_sellingprice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

# Identify categorical features
categorical_features_indices = X_train.select_dtypes(include=['object']).columns.tolist()

# 3. Initialize and Train CatBoostRegressor
model = CatBoostRegressor(
    loss_function='RMSE',
    eval_metric='RMSE',
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    random_seed=RANDOM_STATE,
    early_stopping_rounds=50,
    use_best_model=True,
    verbose=100
)

model.fit(
    X_train, y_train,
    cat_features=categorical_features_indices,
    eval_set=(X_test, y_test),
)

# 4. Predict and Evaluate
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test)

test_mae = mean_absolute_error(y_test_original, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test_original, y_pred))

print(f"\n--- Model Training Complete ---")
print(f"Best iteration: {model.get_best_iteration()}")
print(f"Model Performance (Original Price Scale):")
print(f"Test MAE: ${test_mae:,.2f} (Average absolute error)")
print(f"Test RMSE: ${test_rmse:,.2f} (Penalizes larger errors more heavily)")


--- CatBoost Training and Evaluation ---
0:	learn: 0.8552236	test: 0.8524099	best: 0.8524099 (0)	total: 4.92s	remaining: 1h 21m 55s
100:	learn: 0.2404831	test: 0.2363064	best: 0.2363064 (100)	total: 3m 16s	remaining: 29m 5s
200:	learn: 0.2098081	test: 0.2065204	best: 0.2065204 (200)	total: 6m 11s	remaining: 24m 35s
300:	learn: 0.1959445	test: 0.1934878	best: 0.1934878 (300)	total: 9m	remaining: 20m 56s
400:	learn: 0.1871189	test: 0.1855074	best: 0.1855074 (400)	total: 11m 45s	remaining: 17m 33s
500:	learn: 0.1808832	test: 0.1800449	best: 0.1800449 (500)	total: 14m 29s	remaining: 14m 26s
600:	learn: 0.1760162	test: 0.1758653	best: 0.1758653 (600)	total: 17m 12s	remaining: 11m 25s
700:	learn: 0.1722096	test: 0.1727738	best: 0.1727738 (700)	total: 19m 59s	remaining: 8m 31s
800:	learn: 0.1690547	test: 0.1702738	best: 0.1702738 (800)	total: 22m 40s	remaining: 5m 38s
900:	learn: 0.1663005	test: 0.1681085	best: 0.1681085 (900)	total: 25m 23s	remaining: 2m 47s
999:	learn: 0.1638077	test: 0.16

In [ ]:
# Save the model
MODEL_FILENAME = 'catboost_rag_model.cbm'
model.save_model(
    MODEL_FILENAME,
    format='cbm' # The default binary format for CatBoost
)
print(f"Model saved successfully as binary file: {MODEL_FILENAME}")


# --- How to Load the Model Later ---
from catboost import CatBoostRegressor

# Load the model back into memory
loaded_model = CatBoostRegressor()
loaded_model.load_model(MODEL_FILENAME)

print("Model loaded successfully for future predictions.")